# 🍔 FoodFlow AI — Machine Learning

Notebook de portfólio para análise e previsão em uma plataforma fictícia de delivery.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../data/generated')
orders = pd.read_csv(DATA / 'orders.csv')
reviews = pd.read_csv(DATA / 'reviews.csv')
orders.head()

## 1. Visão do negócio

Perguntas: qual a taxa de cancelamento? quais categorias geram mais receita? como distância, trânsito e preparo se relacionam ao cancelamento?

In [ ]:
orders['gross_value'] = orders['order_value'] + orders['delivery_fee'] - orders['discount']
print('Pedidos:', len(orders))
print('Cancelamentos:', int(orders['cancelled'].sum()))
print('Taxa de cancelamento:', round(orders['cancelled'].mean(), 4))
orders.groupby('category')['gross_value'].sum().sort_values(ascending=False)

## 2. Distribuição de cancelamentos

In [ ]:
orders['cancelled'].value_counts().plot(kind='bar', title='Pedidos cancelados vs. não cancelados')
plt.xlabel('Cancelado')
plt.ylabel('Quantidade')
plt.show()

## 3. Modelo de cancelamento

O pipeline oficial está em `src/train_cancellation.py`. Para reproduzir no Kaggle, copie o pipeline de pré-processamento e compare Precision, Recall, F1 e ROC-AUC.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

features = ['order_hour','category','order_value','delivery_fee','discount','distance_km','preparation_time_min','traffic','contract','payment_method']
X = orders[features]
y = orders['cancelled']
numeric = ['order_hour','order_value','delivery_fee','discount','distance_km','preparation_time_min']
categorical = ['category','traffic','contract','payment_method']
prep = ColumnTransformer([('num', Pipeline([('imp', SimpleImputer(strategy='median')),('scale', StandardScaler())]), numeric),('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),('oh', OneHotEncoder(handle_unknown='ignore'))]), categorical)])
model = Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_train, y_train)
prob = model.predict_proba(X_test)[:,1]
print('ROC-AUC:', round(roc_auc_score(y_test, prob), 4))

## 4. NLP

As avaliações são classificadas por sentimento no baseline e podem ser analisadas com Transformers no módulo opcional do projeto.

In [ ]:
reviews[['rating','review_text']].head(10)

## 5. Conclusão

O projeto conecta dados, ML, NLP, recuperação de informação, RAG e serving em uma única narrativa de engenharia de IA.